In [1]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:.4f}'.format)

TICKERS    = ['ADMA','NTRA','AXON','SHAK','AAPL','MSFT','NVDA','AMZN','GOOG','META']
DATE_START = '2015-01-01'
DATE_END   = '2025-12-31'
RISK_FREE  = 0.05

PROC  = Path('../data/processed')
FINAL = Path('../data/final')
FINAL.mkdir(parents=True, exist_ok=True)

VALID_BUCKETS = [f'{m}_{d}' for m in ['ATM','OTM5','OTM10'] for d in [30,60,90]]
BUCKET_TO_INT = {b: i for i, b in enumerate(VALID_BUCKETS)}
print(f'Buckets ({len(VALID_BUCKETS)}): {VALID_BUCKETS}')
print('Imports OK')

Buckets (9): ['ATM_30', 'ATM_60', 'ATM_90', 'OTM5_30', 'OTM5_60', 'OTM5_90', 'OTM10_30', 'OTM10_60', 'OTM10_90']
Imports OK


## Cell 1 - Load All Raw Data

In [2]:
def load_parquet(path, label):
    df = pd.read_parquet(path)
    print(f'  {label}: {df.shape}')
    return df

print('Loading raw data...')
daily_raw    = load_parquet(PROC / 'daily_adjusted/ALL_daily_adjusted.parquet',  'Daily prices')
income_raw   = load_parquet(PROC / 'fundamentals/ALL_income_statement.parquet',  'Income statement')
balance_raw  = load_parquet(PROC / 'fundamentals/ALL_balance_sheet.parquet',     'Balance sheet')
cashflow_raw = load_parquet(PROC / 'fundamentals/ALL_cash_flow.parquet',         'Cash flow')
options_raw  = load_parquet(PROC / 'options/ALL_options.parquet',                'Options')
overview_raw = pd.read_csv(PROC / 'fundamentals/ALL_overview.csv')
print(f'  Overview: {overview_raw.shape}')

# Parse dates
daily_raw['date']          = pd.to_datetime(daily_raw['date'])
options_raw['trade_date']  = pd.to_datetime(options_raw['trade_date'])
options_raw['expiration']  = pd.to_datetime(options_raw['expiration'])
for df in [income_raw, balance_raw, cashflow_raw]:
    df['fiscalDateEnding'] = pd.to_datetime(df['fiscalDateEnding'])

print('\nAll data loaded and dates parsed.')

Loading raw data...
  Daily prices: (52486, 10)
  Income statement: (781, 27)
  Balance sheet: (773, 39)
  Cash flow: (778, 31)
  Options: (3192217, 20)
  Overview: (10, 55)

All data loaded and dates parsed.


### Data Cleaning

In [3]:
# Null value audit across all raw files 
def null_audit(df, label):
    total     = len(df)
    null_cols = df.isnull().sum()
    null_cols = null_cols[null_cols > 0].sort_values(ascending=False)
    print(f"\n{'='*55}")
    print(f"{label}  —  {df.shape[0]:,} rows * {df.shape[1]} cols")
    print(f"{'='*55}")
    if null_cols.empty:
        print("  ✓ No null values found")
    else:
        print(f"  {len(null_cols)} columns have nulls:\n")
        pct = (null_cols / total * 100).round(1)
        summary = pd.DataFrame({'null_count': null_cols, 'null_pct': pct})
        print(summary.to_string())

null_audit(daily_raw,    "Daily Adjusted")
null_audit(options_raw,  "Options")
null_audit(income_raw,   "Income Statement")
null_audit(balance_raw,  "Balance Sheet")
null_audit(cashflow_raw, "Cash Flow")
null_audit(overview_raw, "Overview")


Daily Adjusted  —  52,486 rows * 10 cols
  ✓ No null values found

Options  —  3,192,217 rows * 20 cols
  ✓ No null values found

Income Statement  —  781 rows * 27 cols
  21 columns have nulls:

                                   null_count  null_pct
interestAndDebtExpense                    781  100.0000
investmentIncomeNet                       781  100.0000
nonInterestIncome                         781  100.0000
comprehensiveIncomeNetOfTax               781  100.0000
depreciation                              781  100.0000
interestIncome                            542   69.4000
netInterestIncome                         511   65.4000
otherNonOperatingIncome                   369   47.2000
netIncomeFromContinuingOperations          52    6.7000
researchAndDevelopment                     28    3.6000
ebitda                                     10    1.3000
depreciationAndAmortization                10    1.3000
interestExpense                             9    1.2000
ebit               

In [4]:
# Drop 100% null columns from each source
income_drop = ['nonInterestIncome','comprehensiveIncomeNetOfTax',
               'interestAndDebtExpense','investmentIncomeNet','depreciation']

balance_drop = ['longTermDebtNoncurrent','otherNonCurrentAssets','deferredRevenue',
                'currentDebt','investments','accumulatedDepreciationAmortizationPPE']

cashflow_drop = ['paymentsForRepurchaseOfCommonStock','paymentsForRepurchaseOfPreferredStock',
                 'proceedsFromOperatingActivities','changeInOperatingLiabilities',
                 'changeInOperatingAssets','proceedsFromSaleOfTreasuryStock',
                 'proceedsFromIssuanceOfPreferredStock',
                 'proceedsFromIssuanceOfLongTermDebtAndCapitalSecuritiesNet',
                 'profitLoss','proceedsFromIssuanceOfCommonStock',
                 'dividendPayoutPreferredStock','proceedsFromRepaymentsOfShortTermDebt',
                 'paymentsForOperatingActivities','paymentsForRepurchaseOfEquity']

# Safe drop — only drop if column exists
income_raw   = income_raw.drop(columns=[c for c in income_drop   if c in income_raw.columns])
balance_raw  = balance_raw.drop(columns=[c for c in balance_drop  if c in balance_raw.columns])
cashflow_raw = cashflow_raw.drop(columns=[c for c in cashflow_drop if c in cashflow_raw.columns])
print('100% null columns dropped.')

100% null columns dropped.


## forward and backward filling null data 

In [5]:
def clean_fundamentals(df, label):
    key_cols = ['symbol', 'fiscalDateEnding', 'reportedCurrency']
    fill_cols = df.columns.difference(key_cols)
    
    df = df.sort_values(['symbol', 'fiscalDateEnding'])
    
    # Forward-fill then backward-fill within each symbol
    df[fill_cols] = (df.groupby('symbol')[fill_cols]
                       .transform(lambda x: x.ffill().bfill()))
    
    # For any still-remaining nulls, fill with column median across all symbols
    for col in fill_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    
    remaining = df.isnull().sum().sum()
    print(f"{label}: {remaining} nulls remaining after clean")
    return df

income_raw   = clean_fundamentals(income_raw,   "Income Statement")
balance_raw  = clean_fundamentals(balance_raw,  "Balance Sheet")
cashflow_raw = clean_fundamentals(cashflow_raw, "Cash Flow")

Income Statement: 0 nulls remaining after clean
Balance Sheet: 0 nulls remaining after clean
Cash Flow: 0 nulls remaining after clean


In [6]:
overview_raw['DividendPerShare'] = overview_raw['DividendPerShare'].fillna(0)
overview_raw['DividendYield']    = overview_raw['DividendYield'].fillna(0)
overview_raw['DividendDate']     = overview_raw['DividendDate'].fillna('N/A')
overview_raw['ExDividendDate']   = overview_raw['ExDividendDate'].fillna('N/A')
overview_raw['PERatio']          = overview_raw['PERatio'].fillna(overview_raw['PERatio'].median())
print("Overview nulls filled.")

Overview nulls filled.


In [7]:
for df, label in [(income_raw,'Income'), (balance_raw,'Balance'),
                  (cashflow_raw,'Cashflow'), (overview_raw,'Overview')]:
    remaining = df.isnull().sum().sum()
    print(f"{label}: {remaining} nulls remaining")

Income: 0 nulls remaining
Balance: 0 nulls remaining
Cashflow: 0 nulls remaining
Overview: 0 nulls remaining


### Daily raw data cleaning

In [8]:
# Check for nulls per ticker — maybe only some tickers have issues
print('\n--- Nulls per ticker ---')
print(daily_raw.groupby('symbol').apply(lambda x: x.isnull().sum().sum()).rename('total_nulls'))


--- Nulls per ticker ---
symbol
AAPL     0
AMZN     0
AVGO     0
GOOG     0
GOOGL    0
META     0
MSFT     0
NVDA     0
TSLA     0
WMT      0
Name: total_nulls, dtype: int64


In [9]:
# Check date coverage per ticker — missing dates = implicit nulls
print('\n--- Date coverage per ticker ---')
coverage = daily_raw.groupby('symbol')['date'].agg(['min','max','count'])
coverage['expected_trading_days'] = 252 * 10  # rough 10 year estimate
coverage['coverage_pct'] = (coverage['count'] / coverage['expected_trading_days'] * 100).round(1)
print(coverage)


--- Date coverage per ticker ---
              min        max  count  expected_trading_days  coverage_pct
symbol                                                                  
AAPL   2000-01-03 2025-12-31   6539                   2520      259.5000
AMZN   2000-01-03 2025-12-31   6539                   2520      259.5000
AVGO   2009-08-06 2025-12-31   4127                   2520      163.8000
GOOG   2014-03-27 2025-12-31   2960                   2520      117.5000
GOOGL  2004-08-19 2025-12-31   5377                   2520      213.4000
META   2012-05-18 2025-12-31   3425                   2520      135.9000
MSFT   2000-01-03 2025-12-31   6539                   2520      259.5000
NVDA   2000-01-03 2025-12-31   6539                   2520      259.5000
TSLA   2010-06-29 2025-12-31   3902                   2520      154.8000
WMT    2000-01-03 2025-12-31   6539                   2520      259.5000


In [10]:
# Check for duplicate rows — same symbol + date appearing twice
dupes = daily_raw.duplicated(subset=['symbol','date'])
print(f'\n--- Duplicate (symbol, date) rows: {dupes.sum()} ---')
if dupes.sum() > 0:
    print(daily_raw[dupes][['symbol','date']].head(10))


--- Duplicate (symbol, date) rows: 0 ---


In [11]:
# Check for zero or negative prices — data quality issue
print('\n--- Zero or negative prices ---')
bad_price = daily_raw[(daily_raw['adj_close'] <= 0) | 
                       (daily_raw['open'] <= 0) | 
                       (daily_raw['close'] <= 0)]
print(f'Rows with zero/negative price: {len(bad_price)}')
if len(bad_price) > 0:
    print(bad_price[['symbol','date','open','close','adj_close']])


--- Zero or negative prices ---
Rows with zero/negative price: 0


In [14]:
# Check for missing trading days — gaps in the date sequence per ticker
print('\n--- Date gaps per ticker (missing trading days) ---')
for ticker in TICKERS:
    df = daily_raw[daily_raw['symbol'] == ticker].sort_values('date')
    date_diffs = df['date'].diff().dt.days.dropna()
    # Normal gap = 1 day (consecutive) or 3 days (weekend)
    # Anything above 5 days is suspicious (holiday = 4 days max)
    big_gaps = date_diffs[date_diffs > 5]
    if len(big_gaps) > 0:
        gap_dates = df.loc[big_gaps.index, 'date']
        print(f'  {ticker}: {len(big_gaps)} gap(s) > 5 days')
        for gap_val, gap_date in zip(big_gaps.values, gap_dates.values):
            print(f'    {pd.Timestamp(gap_date).date()} — {int(gap_val)} day gap')
    else:
        print(f'  {ticker}: ✓ No gaps')


--- Date gaps per ticker (missing trading days) ---
  ADMA: ✓ No gaps
  NTRA: ✓ No gaps
  AXON: ✓ No gaps
  SHAK: ✓ No gaps
  AAPL: 1 gap(s) > 5 days
    2001-09-17 — 7 day gap
  MSFT: 1 gap(s) > 5 days
    2001-09-17 — 7 day gap
  NVDA: 1 gap(s) > 5 days
    2001-09-17 — 7 day gap
  AMZN: 1 gap(s) > 5 days
    2001-09-17 — 7 day gap
  GOOG: ✓ No gaps
  META: ✓ No gaps


In [15]:
#  Summary
print('\n--- Daily prices summary ---')
print(f'Total rows:     {len(daily_raw):,}')
print(f'Tickers:        {daily_raw["symbol"].nunique()}')
print(f'Date range:     {daily_raw["date"].min().date()} to {daily_raw["date"].max().date()}')
print(f'Total nulls:    {daily_raw.isnull().sum().sum()}')
print(f'Columns:        {list(daily_raw.columns)}')


--- Daily prices summary ---
Total rows:     52,486
Tickers:        10
Date range:     2000-01-03 to 2025-12-31
Total nulls:    0
Columns:        ['date', 'symbol', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'dividend', 'split_coeff']


In [16]:
# Local Clean Data Configuration 
CLEAN_DIR = Path('../data/clean')

# Subfolders mirror the raw structure
(CLEAN_DIR / 'daily_adjusted').mkdir(parents=True, exist_ok=True)
(CLEAN_DIR / 'fundamentals').mkdir(parents=True, exist_ok=True)
(CLEAN_DIR / 'options').mkdir(parents=True, exist_ok=True)

In [17]:
# Save Cleaned DataFrames to data/clean/ 
print('\nSaving cleaned files...\n')

files_to_save = {
    'daily_adjusted/ALL_daily_adjusted_clean.parquet' : (daily_raw,    'parquet'),
    'fundamentals/ALL_income_statement_clean.parquet' : (income_raw,   'parquet'),
    'fundamentals/ALL_balance_sheet_clean.parquet'    : (balance_raw,  'parquet'),
    'fundamentals/ALL_cash_flow_clean.parquet'        : (cashflow_raw, 'parquet'),
    'fundamentals/ALL_overview_clean.csv'             : (overview_raw, 'csv'),
    'options/ALL_options_clean.parquet'               : (options_raw,  'parquet'),
}

results = {}
for relative_path, (df, fmt) in files_to_save.items():
    out_path = CLEAN_DIR / relative_path
    try:
        if fmt == 'parquet':
            df.to_parquet(out_path, index=False)
        elif fmt == 'csv':
            df.to_csv(out_path, index=False)
        
        size_mb = out_path.stat().st_size / (1024 * 1024)
        print(f'  ✓ {relative_path}  ({size_mb:.1f} MB)  —  {len(df):,} rows × {df.shape[1]} cols')
        results[relative_path] = True
    except Exception as e:
        print(f'  ✗ FAILED: {relative_path} — {e}')
        results[relative_path] = False


Saving cleaned files...

  ✓ daily_adjusted/ALL_daily_adjusted_clean.parquet  (1.9 MB)  —  52,486 rows × 10 cols
  ✓ fundamentals/ALL_income_statement_clean.parquet  (0.1 MB)  —  781 rows × 22 cols
  ✓ fundamentals/ALL_balance_sheet_clean.parquet  (0.2 MB)  —  773 rows × 33 cols
  ✓ fundamentals/ALL_cash_flow_clean.parquet  (0.1 MB)  —  778 rows × 17 cols
  ✓ fundamentals/ALL_overview_clean.csv  (0.0 MB)  —  10 rows × 55 cols
  ✓ options/ALL_options_clean.parquet  (127.6 MB)  —  3,192,217 rows × 20 cols


In [18]:
# Local summary
success = sum(results.values())
print(f'\n{"="*55}')
print(f'Saved {success}/{len(results)} files to {CLEAN_DIR}')
print(f'\nFolder structure:')
print(f'  data/clean/')
print(f'  ├── daily_adjusted/')
print(f'  │   └── ALL_daily_adjusted_clean.parquet')
print(f'  ├── fundamentals/')
print(f'  │   ├── ALL_income_statement_clean.parquet')
print(f'  │   ├── ALL_balance_sheet_clean.parquet')
print(f'  │   ├── ALL_cash_flow_clean.parquet')
print(f'  │   └── ALL_overview_clean.csv')
print(f'  └── options/')
print(f'      └── ALL_options_clean.parquet')


Saved 6/6 files to ../data/clean

Folder structure:
  data/clean/
  ├── daily_adjusted/
  │   └── ALL_daily_adjusted_clean.parquet
  ├── fundamentals/
  │   ├── ALL_income_statement_clean.parquet
  │   ├── ALL_balance_sheet_clean.parquet
  │   ├── ALL_cash_flow_clean.parquet
  │   └── ALL_overview_clean.csv
  └── options/
      └── ALL_options_clean.parquet


In [19]:
# verify files locally
print(f'\nVerifying files on disk...')
for relative_path in files_to_save:
    p = CLEAN_DIR / relative_path
    if p.exists():
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f'  ✓ {p}  ({size_mb:.1f} MB)')
    else:
        print(f'  ✗ MISSING: {p}')


Verifying files on disk...
  ✓ ../data/clean/daily_adjusted/ALL_daily_adjusted_clean.parquet  (1.9 MB)
  ✓ ../data/clean/fundamentals/ALL_income_statement_clean.parquet  (0.1 MB)
  ✓ ../data/clean/fundamentals/ALL_balance_sheet_clean.parquet  (0.2 MB)
  ✓ ../data/clean/fundamentals/ALL_cash_flow_clean.parquet  (0.1 MB)
  ✓ ../data/clean/fundamentals/ALL_overview_clean.csv  (0.0 MB)
  ✓ ../data/clean/options/ALL_options_clean.parquet  (127.6 MB)


### Upload clean data files to S3

In [20]:
import boto3
import os
from pathlib import Path

In [21]:
# S3 configuration
BUCKET_NAME = 'validex-ml-data'
REGION      = 'us-east-1'

In [22]:
# Local clean data paths — update these to wherever your cleaned files are saved
CLEAN_FILES = {
    'clean_data/daily_adjusted/ALL_daily_adjusted_clean.parquet' : CLEAN_DIR / 'daily_adjusted/ALL_daily_adjusted_clean.parquet',
    'clean_data/fundamentals/ALL_income_statement_clean.parquet' : CLEAN_DIR / 'fundamentals/ALL_income_statement_clean.parquet',
    'clean_data/fundamentals/ALL_balance_sheet_clean.parquet'    : CLEAN_DIR / 'fundamentals/ALL_balance_sheet_clean.parquet',
    'clean_data/fundamentals/ALL_cash_flow_clean.parquet'        : CLEAN_DIR / 'fundamentals/ALL_cash_flow_clean.parquet',
    'clean_data/fundamentals/ALL_overview_clean.csv'             : CLEAN_DIR / 'fundamentals/ALL_overview_clean.csv',
    'clean_data/options/ALL_options_clean.parquet'               : CLEAN_DIR / 'options/ALL_options_clean.parquet',
}

In [23]:
# Upload function
def upload_to_s3(local_path, s3_key, bucket, s3_client):
    local_path = Path(local_path)
    if not local_path.exists():
        print(f'  SKIPPED — file not found: {local_path}')
        return False
    
    file_size_mb = local_path.stat().st_size / (1024 * 1024)
    print(f'  Uploading {local_path.name} ({file_size_mb:.1f} MB) → s3://{bucket}/{s3_key}')
    
    try:
        s3_client.upload_file(
            Filename = str(local_path),
            Bucket   = bucket,
            Key      = s3_key,
            Callback = None
        )
        print(f'  ✓ Done')
        return True
    except Exception as e:
        print(f'  ✗ FAILED: {e}')
        return False

In [24]:
# Run upload
print('Connecting to S3...')
s3 = boto3.client('s3', region_name=REGION)

# Verify bucket is accessible
try:
    s3.head_bucket(Bucket=BUCKET_NAME)
    print(f'✓ Connected to bucket: {BUCKET_NAME}\n')
except Exception as e:
    raise ConnectionError(f'Cannot access bucket {BUCKET_NAME}: {e}')

# Upload all files
print(f'Uploading {len(CLEAN_FILES)} files to s3://{BUCKET_NAME}/clean_data/\n')
results = {}
for s3_key, local_path in CLEAN_FILES.items():
    results[s3_key] = upload_to_s3(local_path, s3_key, BUCKET_NAME, s3)

Connecting to S3...
✓ Connected to bucket: validex-ml-data

Uploading 6 files to s3://validex-ml-data/clean_data/

  Uploading ALL_daily_adjusted_clean.parquet (1.9 MB) → s3://validex-ml-data/clean_data/daily_adjusted/ALL_daily_adjusted_clean.parquet
  ✓ Done
  Uploading ALL_income_statement_clean.parquet (0.1 MB) → s3://validex-ml-data/clean_data/fundamentals/ALL_income_statement_clean.parquet
  ✓ Done
  Uploading ALL_balance_sheet_clean.parquet (0.2 MB) → s3://validex-ml-data/clean_data/fundamentals/ALL_balance_sheet_clean.parquet
  ✓ Done
  Uploading ALL_cash_flow_clean.parquet (0.1 MB) → s3://validex-ml-data/clean_data/fundamentals/ALL_cash_flow_clean.parquet
  ✓ Done
  Uploading ALL_overview_clean.csv (0.0 MB) → s3://validex-ml-data/clean_data/fundamentals/ALL_overview_clean.csv
  ✓ Done
  Uploading ALL_options_clean.parquet (127.6 MB) → s3://validex-ml-data/clean_data/options/ALL_options_clean.parquet
  ✓ Done


In [26]:
# Summary
success = sum(results.values())
failed  = len(results) - success
print(f'\n{"="*55}')
print(f'Upload complete: {success}/{len(results)} files successful')
if failed > 0:
    print(f'Failed files:')
    for key, ok in results.items():
        if not ok:
            print(f'  ✗ {key}')
print(f'\nS3 folder structure:')
print(f'  s3://{BUCKET_NAME}/')
print(f'  └── clean_data/')
print(f'      ├── daily_adjusted/')
print(f'      │   └── ALL_daily_adjusted_clean.parquet')
print(f'      ├── fundamentals/')
print(f'      │   ├── ALL_income_statement_clean.parquet')
print(f'      │   ├── ALL_balance_sheet_clean.parquet')
print(f'      │   ├── ALL_cash_flow_clean.parquet')
print(f'      │   └── ALL_overview_clean.csv')
print(f'      └── options/')
print(f'          └── ALL_options_clean.parquet')


Upload complete: 6/6 files successful

S3 folder structure:
  s3://validex-ml-data/
  └── clean_data/
      ├── daily_adjusted/
      │   └── ALL_daily_adjusted_clean.parquet
      ├── fundamentals/
      │   ├── ALL_income_statement_clean.parquet
      │   ├── ALL_balance_sheet_clean.parquet
      │   ├── ALL_cash_flow_clean.parquet
      │   └── ALL_overview_clean.csv
      └── options/
          └── ALL_options_clean.parquet
